<a href="https://colab.research.google.com/github/sudedoruk/DETR_bitirme/blob/main/detr_bitirme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DETR

In [ ]:
import torch #PyThorch:çok boyutlu matris işlemler
from torch import nn # neural network(nn) modülü
                     #Sinir ağı katmanlarını (Linear, Conv2d, Transformer, BatchNorm, vb.)
                     #ve modelleri (nn.Module) tanımlamak için
from torchvision.models import resnet50 #DETRde özellik haritaları(backone) çıkarmak için
  #hazır cnn modelleri,veri setleri var

class DETR(nn.Module):#nn.module sınıfından detr sınıfı türetildi
    #parametreler    tahmin edilecek sınıf sayısı,transformerın ara katman boyutu
    def __init__(self, num_classes, hidden_dim, nheads,num_encoder_layers, num_decoder_layers):
                                              #MHAdaki head sayısı,encoder/decoder katman sayıları
        super().__init__()#..module fonk.u çağrılır

        # Backbone(özellik haritası): ResNet-50'in konvolüsyon katmanları (tam bağlı katmanlar(resnetin sonundaki sınıflandırmra katmanları) çıkarılıyo(-2)görme kısmı kalıyo)
        #her piksel için feature map oluşturuluyo(grid)
        self.backbone = nn.Sequential(*list(resnet50(pretrained=True).children())[:-2])
        #resnet modelini çağırıp katmanları listeye atıp son iki katmani çıkarıyo(detr kullanmıyo çünkü)en son da dizi model haline getiriyo katmanları

        # Özellik haritasını transformer boyutuna indirgeme
        self.conv = nn.Conv2d(2048, hidden_dim, kernel_size=1) #2048->256
        #resnetin çıkışı 2048 kanal,transfırmer 256 boyutlu vektörle çalışır(hidden_dim=256)

        # Transformer: encoder + decoder görüntüyü inceler karar verir
        #girdiler:pozisyon kodlu özellik haritası+sorgu vektörleri
        self.transformer = nn.Transformer(
            d_model=hidden_dim,
            nhead=nheads,
            num_encoder_layers=num_encoder_layers,#resnetten gelen özellikler işlenir
            num_decoder_layers=num_decoder_layers#nesne sorgularıyla sonuç üretir.
        )

        #Çıktı başlıkları: sınıf ve kutu tahmini
                          #nn.Linear(giriş sayısı,çıkış sayısı)x girişten y çıkış üreten tam bağlı katman
        self.linear_class = nn.Linear(hidden_dim, num_classes + 1)  # her nesne sorgusu için sınıf tahmin,+1: "no object" sınıfı
        self.linear_bbox = nn.Linear(hidden_dim, 4)  # aynı sorgu için 4 değer üretiyo-> (cx, cy, w, h)kutu kordinatları(merkez x,y,weight,hight)

        #Öğrenilebilir sorgu vektörleri oluşturma ve öğrenilebilir hale getirme(nn.Parameter)
        #(100 tane kutu tahmini,sorgu boyutu(transformerın iç boyutuyla aynı olmalı))
        self.query_pos = nn.Parameter(torch.rand(100, hidden_dim))

        # Pozisyon kodları (satır ve sütun için ayrı ayrı),konum bilgilerini modele aktarıyo
        #transformer pikselin nerde olduğunu bilmez(sıra bilgisiyle çalışıyo) o yüzden her satır/sütuna konum vektörü eklenir ve nesnenin hangi bölgede olduğunu öğrenir
        #resnet 800x1200lük görüntüyünişlerken 32 adımlıkçıkış üretir
              # 800/32=25,1200/32=37 -->> çıktı feature map boyutu
              #yani 50 max grid boyutu 25x37 için yeterli ,giriş daha büyükse sadec H,W kısmı alınıyo(aşağıda)
        self.row_embed = nn.Parameter(torch.rand(50, hidden_dim // 2))
        self.col_embed = nn.Parameter(torch.rand(50, hidden_dim // 2))
                                                    #256 boyutu satır ve sütuna bölüyo 128
                                                    #görüntüler 2 boyutlu,transformer da 2D algılamalı
    #feature map [B,C,H,W]->Görüntü sayısı(batch size),kanal sayısı,yükseklik,genişlik
    def forward(self, inputs):
        #Görüntüden özellik çıkarımı
        x = self.backbone(inputs)        # [B, 2048, H, W] görüntüden özellik haritası çıkarılır 2048 kanal
        x = self.conv(x)                 # [B, hidden_dim, H, W] 256 kanala indirger
        H, W = x.shape[-2:]  #.shape x tensorünğn boyutlarını verir.-2->sondan iki boyutu alır(H,W)

        #Pozisyon kodlaması (H x W boyutunda)
        #vektörler birleştiriliyo,2Dlu konum bilgisi(embedding)->pos[i, j] = concat(row_embed[i], col_embed[j])
        pos = torch.cat([
            self.col_embed[:W].unsqueeze(0).repeat(H, 1, 1),  # [H, W, C//2]   unsqueeze:boyut ekler,satır,sütun vs(satır+),repeat:o boyuttaki veriyi çoğaltır(H kez tekrarlandı)
            self.row_embed[:H].unsqueeze(1).repeat(1, W, 1)   # [H, W, C//2]             sütun+,her sütunda satır embeddingi aynı kalıyo
        ], dim=-1).flatten(0, 1).unsqueeze(1)                 # [HW, 1, C]  flatten():2 boyutu tek boyuta indiriyo(grid sıralı hale geldi)
                                                                            #unsqueeze(1) ortaya bach ekseni eklendi(kaç veri işleyeceği)
        #Transformer girişi: pozisyon kodu + özellik haritası
        src = x.flatten(2).permute(2, 0, 1)                   # [HW, B, C]  2.ve 3. boyutları birleştirip sırasını değiştirme
        tgt = self.query_pos.unsqueeze(1)                    # [Q, 1, C]    decoder girişi,nesne sorguları,unsqueeze(1)->transformer için zorunlu
                                                                            #birden fazla görüntü için .repeat(,,)
        h = self.transformer(src + pos, tgt)                 # [Q, B, C]    decoder çıktısı  [query,batch,transformer çıkış boyutu]  [100,1,256]
                            #özellik ve pozisyon bilgisi
        #Çıktılar: sınıf ve kutu tahmini
        return self.linear_class(h), self.linear_bbox(h).sigmoid()  #tahmin sonucunun hangi sınıfa ait olduğu,konumu(kutu tahmini),kordinatları[0,1] aralığına şıkıştırma(sigmoid)

#Model örneği ve test girişi   #detr sınıfının örneği oluşturuluyo
detr = DETR(num_classes=91, hidden_dim=256, nheads=8,    #nesne sınıfı sayısı(coco için 91)
            num_encoder_layers=6, num_decoder_layers=6)
detr.eval() #eğitim bitti test başladı demek.modeli sabit hale geitirir.rastgelelik yok artık..
            #dropout kapalı,batch norm donduruldu
            #dropout:testte rastgeleliği önlemek için nöronları(fonk.lar) aktif tutar
            #batch norm:her batchteki ortalama ve varyansları güncellemez ,sbt değer kullanılır
#Örnek giriş görüntüsü (batch size = 1, 3 kanal, 800x1200)
inputs = torch.randn(1, 3, 800, 1200) #1 adet 3 kanallı 800x1200 boyutunda rastgele değerli tensör üretiliyo

#Tahmin
logits, bboxes = detr(inputs)   #her sorgu için tahmin edilen sınıf ve kutu değerleri

özet:

*   özellik çıkarımı(CNN),feature map
*   pozisyon embedding
*   transformer encoder
*   transformer decoder
*   test+sınıf ve kutu bilgileri






# ESO-DETR

In [ ]:
import torch #PyThorch:çok boyutlu matris işlemler
from torch import nn # neural network(nn) modülü
                     #Sinir ağı katmanlarını (Linear, Conv2d, Transformer, BatchNorm, vb.)
                     #ve modelleri (nn.Module) tanımlamak için
from torchvision.models import resnet18 #DETRde özellik haritaları(backone) çıkarmak için
  #hazır cnn modelleri,veri setleri var

class ESODETR(nn.Module):#nn.module sınıfından detr sınıfı türetildi
    #parametreler    tahmin edilecek sınıf sayısı,transformerın ara katman boyutu
    def __init__(self, num_classes, hidden_dim, nheads,num_encoder_layers, num_decoder_layers):
                                              #MHAdaki head sayısı,encoder/decoder katman sayıları
        super(GHSA_Block, self).__init__()
        self.backbone = nn.Sequential(*list(resnet18(pretrained=True).children())[:-2])

        self.norm = nn.BatchNorm2d(dim)
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

        # Q,K,V projeksiyonları (tek başlı)
        self.q = nn.Conv2d(dim, dim, kernel_size=1)
        self.k = nn.Conv2d(dim, dim, kernel_size=1)
        self.v = nn.Conv2d(dim, dim, kernel_size=1)

        # Gating (CGLU)
        self.w_g = nn.Conv2d(dim, dim, kernel_size=1)
        self.w_x = nn.Conv2d(dim, dim, kernel_size=1)

        self.proj = nn.Conv2d(dim, dim, kernel_size=1)
        self.scale = dim ** 0.5

    def forward(self, x):
        B, C, H, W = x.size()
        identity = x

        # 1️⃣ BatchNorm + Depthwise Conv
        x = self.norm(x)
        dw = self.dwconv(x)

        # 2️⃣ Q, K, V oluştur
        q = self.q(x).flatten(2).permute(0, 2, 1)    # [B, HW, C]
        k = self.k(x).flatten(2)                     # [B, C, HW]
        v = self.v(x).flatten(2).permute(0, 2, 1)    # [B, HW, C]

        # 3️⃣ Attention hesapla
        attn = torch.bmm(q, k) / self.scale          # [B, HW, HW]
        attn = F.softmax(attn, dim=-1)
        attn_out = torch.bmm(attn, v)                # [B, HW, C]
        attn_out = attn_out.transpose(1, 2).reshape(B, C, H, W)

        # 4️⃣ Gating (CGLU)
        gate = torch.sigmoid(self.w_g(x))
        gated_out = self.w_x(x) * gate

        # 5️⃣ Toplama (Hybrid yapı)
        out = dw + attn_out + gated_out
        out = self.proj(out)
        return out + identity